In [ ]:
%%capture
!pip install tqdm
!pip install ffmpeg-python 
!pip install opencv-python
!pip install kaggle

In [ ]:
import os
import shutil
import cv2
import json
import logging
import math
import subprocess
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import ffmpeg

class VideoProcess:
    def __init__(
        self,
        input_path: str,
        output_path: str,
        kaggle_api: dict,
        base_dataset_name: str,
        progress_info: dict = {},
        cleanup_after_zip: bool = True,   # <--- mới
    ):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.progress_file_path = Path("progress.json")
        self.progress_info = progress_info or {}

        # Kaggle
        self.kaggle_username = (kaggle_api or {}).get("username")
        self.kaggle_key = (kaggle_api or {}).get("key")
        if not self.kaggle_username or not self.kaggle_key:
            raise ValueError("Kaggle API 'username' và 'key' không được để trống.")
        self._setup_kaggle_credentials(self.kaggle_username, self.kaggle_key)

        # tên dataset SẼ DÙNG NGUYÊN VẸN (không thêm batch suffix)
        self.base_dataset_name = base_dataset_name

        # nơi gom zip để upload 1 lượt sau khi xử lý xong
        self.upload_dir = self.output_path / "upload"
        self.cleanup_after_zip = cleanup_after_zip

        logging.info(f"Dọn dẹp thư mục đầu ra '{self.output_path}'...")
        shutil.rmtree(self.output_path, ignore_errors=True)
        self.output_path.mkdir(parents=True, exist_ok=True)
        self.upload_dir.mkdir(exist_ok=True)

        if not self.input_path.is_dir():
            raise NotADirectoryError(f"Đường dẫn đầu vào không tồn tại hoặc không phải là thư mục: {self.input_path}")
        logging.info(f"Khởi tạo VideoProcess thành công. Output sẽ được lưu tại: '{self.output_path}'")

    def _setup_kaggle_credentials(self, username: str, key: str):
        kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(kaggle_json_path, "w") as f:
            json.dump({"username": username, "key": key}, f)
        os.chmod(kaggle_json_path, 0o600)
        logging.info("Đã cấu hình Kaggle credentials.")

    def get_video_files(self, file_extension: str = '.mp4') -> list:
        file_list = sorted([Path(root) / f for root, _, files in os.walk(self.input_path) for f in files if f.endswith(file_extension)])
        logging.info(f"Tìm thấy tổng cộng {len(file_list)} file video với phần mở rộng '{file_extension}'.")

        if not self.progress_info:
            self.progress_info = {
                "total_files": len(file_list),
                "start_index": 0,
                "end_index": len(file_list)
            }
            with open(self.progress_file_path, 'w') as f:
                json.dump(self.progress_info, f, indent=4)
            logging.info(f"Tạo mới file tiến trình: {self.progress_file_path}")

        return file_list[self.progress_info["start_index"]:self.progress_info["end_index"]]

    @staticmethod
    def _convert_to_hls_worker(task: tuple) -> tuple:
        input_video, hls_output_dir, segment_duration = task
        try:
            hls_output_dir.mkdir(parents=True, exist_ok=True)
            playlist_path = hls_output_dir / 'playlist.m3u8'
            segment_filename = hls_output_dir / 'segment-%05d.ts'

            (
                ffmpeg.input(str(input_video))
                .output(
                    str(playlist_path),
                    format='hls',
                    hls_time=segment_duration,
                    hls_list_size=0,
                    hls_segment_filename=str(segment_filename),
                    c='copy'
                )
                .run(capture_stdout=True, capture_stderr=True, quiet=True)
            )
            return (input_video.name, True, f"Đã tạo HLS tại: {hls_output_dir}")
        except ffmpeg.Error as e:
            error_message = e.stderr.decode('utf-8', errors='ignore').strip()
            shutil.rmtree(hls_output_dir, ignore_errors=True)
            return (input_video.name, False, error_message)
        except Exception as e:
            shutil.rmtree(hls_output_dir, ignore_errors=True)
            return (input_video.name, False, str(e))

    def _get_dir_size_in_gb(self, dir_path: Path) -> float:
        total_size = sum(f.stat().st_size for f in dir_path.glob('**/*') if f.is_file())
        return total_size / (1024 ** 3)

    # -------------- Kaggle helpers --------------
    @staticmethod
    def _slugify(name: str) -> str:
        s = name.lower().strip().replace(' ', '-').replace('_', '-')
        s = ''.join(ch for ch in s if ch.isalnum() or ch == '-')
        while '--' in s:
            s = s.replace('--', '-')
        return s.strip('-')

    def _write_dataset_metadata(self, dataset_name: str):
        dataset_slug = self._slugify(dataset_name)
        dataset_id = f"{self.kaggle_username}/{dataset_slug}"
        meta = {
            "title": dataset_name,
            "id": dataset_id,
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(self.upload_dir / 'dataset-metadata.json', 'w', encoding='utf-8') as f:
            json.dump(meta, f, indent=4, ensure_ascii=False)
        return dataset_id

    def _create_or_update_dataset(self, dataset_name: str, message: str):
        dataset_id = self._write_dataset_metadata(dataset_name)
        cmd_create = ["kaggle", "datasets", "create", "-p", str(self.upload_dir), "--dir-mode", "zip"]
        logging.info(f"Tạo dataset mới: {dataset_id}")
        try:
            subprocess.run(cmd_create, check=True, capture_output=True, text=True, encoding='utf-8')
            logging.info(f"Đã tạo dataset {dataset_id}")
        except subprocess.CalledProcessError as e:
            stderr = (e.stderr or "") + (e.stdout or "")
            if "Already exists" in stderr or "Conflict" in stderr or "409" in stderr:
                logging.info(f"Dataset đã tồn tại. Tạo version mới cho {dataset_id}")
                cmd_version = [
                    "kaggle", "datasets", "version",
                    "-p", str(self.upload_dir),
                    "-m", message,
                    "--dir-mode", "zip"
                ]
                subprocess.run(cmd_version, check=True, capture_output=True, text=True, encoding='utf-8')
                logging.info(f"Đã tạo version mới cho {dataset_id}")
            else:
                logging.error(f"Lỗi tạo dataset: {stderr}")
                raise

    # -------------- Zip & Upload 1 LẦN --------------
    def _zip_all_hls_outputs(self):
        """
        Zip TẤT CẢ thư mục HLS của các video đã xử lý vào self.upload_dir.
        Mỗi video -> 1 file zip. Nếu self.cleanup_after_zip=True thì xoá thư mục gốc.
        """
        shutil.rmtree(self.upload_dir, ignore_errors=True)
        self.upload_dir.mkdir(exist_ok=True)

        zipped = 0
        for sub in sorted(self.output_path.iterdir()):
            if sub.name == "upload":
                continue
            if sub.is_dir() and (sub / "playlist.m3u8").exists():
                base = self.upload_dir / sub.name
                shutil.make_archive(str(base), 'zip', str(sub))
                if self.cleanup_after_zip:
                    shutil.rmtree(sub, ignore_errors=True)
                zipped += 1
        return zipped

    # -------------- Main pipeline --------------
    def process_videos_in_batches(self, batch_size: int, segment_duration: int = 30, file_extension: str = '.mp4'):
        list_video = self.get_video_files(file_extension)
        if not list_video:
            logging.warning("Không tìm thấy video nào để xử lý.")
            return

        num_batches = math.ceil(len(list_video) / batch_size)
        logging.info(f"Tổng số video sẽ được chia thành {num_batches} lô, mỗi lô có tối đa {batch_size} video.")
        all_results = []

        # Chỉ CONVERT theo lô, KHÔNG zip/không upload ở giữa chừng
        for i in range(num_batches):
            print(f"\n{'='*20} BẮT ĐẦU LÔ {i + 1}/{num_batches} {'='*20}")

            start_index = i * batch_size
            end_index = start_index + batch_size
            current_batch_videos = list_video[start_index:end_index]

            tasks = []
            for video_path in current_batch_videos:
                output_dir_for_video = self.output_path / Path(video_path).with_suffix('').name
                tasks.append((video_path, output_dir_for_video, segment_duration))

            with Pool(max(1, cpu_count()-1)) as p:
                iterator = p.imap_unordered(self._convert_to_hls_worker, tasks)
                batch_results = list(tqdm(iterator, total=len(current_batch_videos), desc=f"Lô {i+1}/{num_batches}"))
                all_results.extend(batch_results)

            hls_size_gb = self._get_dir_size_in_gb(self.output_path)
            print(f"Hoàn tất HLS Lô {i + 1}. Dung lượng tạm thời: {hls_size_gb:.4f} GB.")

        # ======= TỚI ĐÂY MỚI NÉN VÀ UPLOAD 1 LẦN =======
        print(f"\n{'='*20} NÉN TOÀN BỘ OUTPUT {'='*20}")
        zipped_count = self._zip_all_hls_outputs()
        total_zip_gb = self._get_dir_size_in_gb(self.upload_dir)
        print(f"Đã zip {zipped_count} thư mục HLS vào upload/. Tổng dung lượng zip: {total_zip_gb:.4f} GB.")

        if total_zip_gb == 0:
            print("Cảnh báo: Không có dữ liệu để upload. Kết thúc.")
            return

        dataset_name = self.base_dataset_name  # dùng nguyên vẹn
        upload_message = f"Tải lên {dataset_name}: toàn bộ HLS (segment_duration={segment_duration}s)."
        try:
            self._create_or_update_dataset(dataset_name, upload_message)
            print(f"✅ Upload Kaggle Dataset thành công: {dataset_name}")
        except subprocess.CalledProcessError as e:
            print("❌ Lỗi upload Kaggle Dataset.")
            print("Stdout:", e.stdout)
            print("Stderr:", e.stderr)

        print(f"\n{'='*20} HOÀN TẤT TOÀN BỘ {'='*20}")
        success_count = sum(1 for _, success, _ in all_results if success)
        failed_files = [(name, msg) for name, success, msg in all_results if not success]

        print(f"Tổng số file đã xử lý: {len(all_results)}")
        print(f"✅ Thành công: {success_count}")
        print(f"❌ Thất bại: {len(failed_files)}")
        if failed_files:
            print("\n--- DANH SÁCH FILE LỖI ---")
            for filename, error in failed_files:
                print(f"  - File: {filename}\n    Lỗi: {error[:250]}...")
        print("="*58)


In [ ]:
# ===================== CẤU HÌNH CHÍNH =====================
INPUT_DATASET_PATH = '/kaggle/input/lucifer-v6'
OUTPUT_HLS_PATH   = '/kaggle/temp'
BATCH_SIZE        = 95
SEGMENT_DURATION  = 10

KAGGLE_API = {"username":"trandiep2105","key":"e4b793444a83cefa3b1aa1a0957b8f30"}

BASE_DATASET_NAME = "lucifer-v6-hls"  # sẽ DÙNG NGUYÊN VẸN tên này

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

if __name__ == "__main__":
    try:
        processor = VideoProcess(
            input_path=INPUT_DATASET_PATH,
            output_path=OUTPUT_HLS_PATH,
            kaggle_api=KAGGLE_API,
            base_dataset_name=BASE_DATASET_NAME,
            cleanup_after_zip=True,  # đặt False nếu muốn giữ lại thư mục HLS sau khi zip
        )
        processor.process_videos_in_batches(
            batch_size=BATCH_SIZE,
            segment_duration=SEGMENT_DURATION,
            file_extension='.mp4'
        )
    except Exception as e:
        logging.error(f"Một lỗi nghiêm trọng đã xảy ra trong chương trình chính: {e}")
